# Finite Difference Methods for Option Pricing

From-scratch implementations of explicit, implicit, and Crank-Nicolson finite difference schemes for the Black-Scholes PDE, including American options.

**Outline**
1. Motivation -- when formulas aren't enough
2. The Black-Scholes PDE
3. Grid setup
4. Explicit finite difference
5. Stability analysis (CFL condition)
6. Implicit finite difference
7. Crank-Nicolson scheme
8. European option pricing comparison
9. American options (PSOR)
10. Greeks from the grid
11. Convergence analysis
12. References
> **Prerequisites:** You should be familiar with the Black-Scholes formula for European options and understand that it arises from solving a partial differential equation (PDE). No prior knowledge of numerical methods is assumed — we build everything from first principles.


---

## 1. Sometimes There's No Formula

If you have studied the Black-Scholes model, you know that there is a beautiful closed-form formula for
the price of a European call or put option. You plug in five numbers -- stock price, strike, time to
expiry, risk-free rate, and volatility -- and out pops a price. It is elegant, and it won a Nobel Prize.

But here is the uncomfortable truth: **most real-world derivatives do not have a closed-form solution.**

Consider these common situations where the Black-Scholes formula simply does not apply:

- **American options** can be exercised at any time before expiry. The holder faces a continuous decision:
  "Should I exercise now, or wait?" There is no known closed-form solution for an American put.
- **Barrier options** knock in or knock out when the stock price hits a certain level. The path-dependence
  makes analytical pricing much harder.
- **Asian options** have a payoff that depends on the *average* stock price over some period, not just the
  final price.
- **Options under local volatility models** where $\sigma = \sigma(S, t)$ varies with the stock price
  and time -- realistic but analytically intractable.

For all of these, we need **numerical methods**. There are three main families:

| Method | Idea | Best for |
|:-------|:-----|:---------|
| **Finite differences** | Solve the PDE on a grid | 1-2 dimensional problems, Greeks |
| **Monte Carlo** | Simulate random paths, average payoffs | High-dimensional problems |
| **Binomial/trinomial trees** | Discrete approximation to the stock process | Pedagogical clarity, American options |

This notebook focuses on **finite difference methods** -- the workhorse of derivatives pricing in
practice. The idea is simple in principle:

1. Write down the PDE that the option price satisfies (the Black-Scholes PDE).
2. Divide the problem domain into a grid of small boxes.
3. Approximate the smooth derivatives in the PDE with discrete differences on the grid.
4. Solve the resulting system of equations, stepping backward from expiry to today.

> **Key Concept:** Think of finite differences as approximating a smooth curve with a connect-the-dots
> drawing. More dots (a finer grid) gives a better approximation, but costs more computation. The art is
> choosing the right balance between accuracy and speed.

### The Heat Equation Analogy

Here is a remarkable fact that connects finance to physics. The Black-Scholes PDE, after a change of
variables, becomes the **heat equation** -- the very same equation that describes how temperature
diffuses through a metal bar.

The one-dimensional heat equation is:

$$\frac{\partial u}{\partial \tau} = \frac{\partial^2 u}{\partial x^2}$$

Imagine a metal bar that is initially hot in some places and cold in others. As time passes, heat
flows from hot spots to cold spots, and the temperature profile smooths out. The heat equation
describes exactly how this happens.

Now here is the connection. If we make the substitutions $S = K e^x$ and $t = T - \frac{2\tau}{\sigma^2}$
and factor out a discount term, the Black-Scholes PDE transforms into the heat equation. So
**pricing an option is mathematically equivalent to tracking how heat diffuses!**

This is not just a curiosity. It means that:

- Physicists and engineers have been solving this equation for over 200 years.
- All the numerical techniques developed for heat conduction -- explicit schemes, implicit schemes,
  Crank-Nicolson -- carry over directly to option pricing.
- The intuition from physics helps: option value "diffuses" from the known payoff at expiry backward
  through time, just as heat diffuses forward through space.

> **Important:** The Black-Scholes PDE is solved *backward* in time (from expiry to today), while the
> heat equation is typically solved *forward*. The change of variables above reverses the time
> direction, which is why the two are equivalent.> **Intuition:** Think of the option value as "heat" that diffuses backwards through time from the known payoff at expiry. Just as heat flows from hot to cold regions, option value propagates from the boundary conditions (payoff at $T$) back to the present. The diffusion coefficient is $\frac{1}{2}\sigma^2 S^2$ — higher volatility means faster diffusion, which means more "spread" of option value into out-of-the-money regions.


### The Three Finite Difference Schemes at a Glance

Before diving into the details, here is a roadmap of the three methods we will implement. Each one
approximates the same PDE, but differs in *how* it handles the time derivative.

| Scheme | How it works | Pros | Cons |
|:-------|:-------------|:-----|:-----|
| **Explicit** | Compute new values directly from old ones -- no equation solving | Simple to code, fast per step | Can blow up if time step is too large (conditionally stable) |
| **Implicit** | Solve a system of equations at each step | Always stable, any time step works | Only first-order accurate in time; more work per step |
| **Crank-Nicolson** | Average of explicit and implicit | Stable *and* second-order accurate | Slight oscillation risk near payoff kinks |

We will implement all three from scratch, compare them on a European call, and then extend to
American options.

---

## 2. Setup and Imports

We use NumPy for matrix operations, SciPy for the normal distribution (needed for Black-Scholes
analytical prices), and Matplotlib for visualization. All finite difference schemes are implemented
from scratch -- no PDE solver libraries.

The cell below also sets up some plotting defaults and color constants that we will use throughout
the notebook.

In [ ]:
%matplotlib inline
import numpy as np
from scipy import stats, optimize, linalg
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

SEED = 42
rng = np.random.default_rng(SEED)

ATOL = 1e-10
RTOL = 1e-6

PRIMARY   = 'steelblue'
SECONDARY = 'coral'
TERTIARY  = 'seagreen'
ACCENT    = 'gold'
plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 12, 'axes.grid': True, 'grid.alpha': 0.3})

We now have our environment ready. `numpy` will handle all the array arithmetic, `scipy.stats` gives
us the cumulative normal distribution for the analytical Black-Scholes formula, and `matplotlib`
will produce all our plots.> **Note:** Every solver in this notebook is built from scratch — no PDE libraries. This means you can inspect exactly how the tridiagonal systems are assembled, how boundary conditions are applied, and where the stability constraints come from.


---

## 3. The Black-Scholes PDE

The starting point for everything in this notebook is the **Black-Scholes partial differential equation**.
Under the standard assumptions (geometric Brownian motion for the stock, constant $r$ and $\sigma$, no
dividends, frictionless markets), the price $V(S, t)$ of any European derivative satisfies:

$$\frac{\partial V}{\partial t} + \frac{1}{2}\sigma^2 S^2 \frac{\partial^2 V}{\partial S^2} + r S \frac{\partial V}{\partial S} - rV = 0$$

with **terminal condition** $V(S, T) = \phi(S)$, where $\phi$ is the payoff function. For a call,
$\phi(S) = \max(S - K, 0)$; for a put, $\phi(S) = \max(K - S, 0)$.

### Reading the PDE Term by Term

Each term in the PDE has a financial interpretation. Let us walk through them one at a time:

| Term | Mathematical role | Financial meaning |
|:-----|:------------------|:------------------|
| $\frac{\partial V}{\partial t}$ | Time derivative | **Time decay** -- how much value the option loses as time passes (theta) |
| $\frac{1}{2}\sigma^2 S^2 \frac{\partial^2 V}{\partial S^2}$ | Diffusion (second-order spatial) | **Gamma/convexity** -- the curvature of the option price. This is what makes options valuable: convexity means you gain more from upside than you lose from downside |
| $r S \frac{\partial V}{\partial S}$ | Convection (first-order spatial) | **Risk-neutral drift** -- under the risk-neutral measure, the stock grows at rate $r$, not $\mu$ |
| $-rV$ | Reaction/discounting | **Discounting** -- a dollar received in the future is worth less today |

The PDE says that these four effects must perfectly balance at every point $(S, t)$. If you know the
option value at expiry (the payoff), you can work backward through time to find today's price.

> **Key Concept:** The Black-Scholes PDE is a **parabolic PDE** -- the same class as the heat equation.
> This means it has a natural direction of information flow (backward from expiry), and the solution
> smooths out as you move away from the terminal condition. This is why finite difference methods
> work so well: they follow this natural information flow.

### Why We Need the Analytical Formula

For European vanilla options, the Black-Scholes PDE has an exact analytical solution. We will
implement this closed-form formula not because we need it for pricing (we are building numerical
methods!), but because it gives us a **ground truth** to validate our finite difference schemes
against. If our numerical solution does not converge to the analytical answer as we refine the grid,
something is wrong.

The Black-Scholes call price is:

$$C = S_0 \, \Phi(d_1) - K e^{-rT} \Phi(d_2)$$

where $d_1 = \frac{\ln(S_0/K) + (r + \sigma^2/2)T}{\sigma\sqrt{T}}$, $d_2 = d_1 - \sigma\sqrt{T}$,
and $\Phi$ is the standard normal CDF.

The put price follows from put-call parity: $P = K e^{-rT}\Phi(-d_2) - S_0 \Phi(-d_1)$.

The next cell implements these formulas and prints reference prices that we will use throughout the
notebook.

In [ ]:
def bs_call(S, K, T, r, sigma):
    """Black-Scholes European call price."""
    if T < 1e-14:
        return np.maximum(S - K, 0)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * stats.norm.cdf(d1) - K * np.exp(-r * T) * stats.norm.cdf(d2)

def bs_put(S, K, T, r, sigma):
    """Black-Scholes European put price."""
    if T < 1e-14:
        return np.maximum(K - S, 0)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return K * np.exp(-r * T) * stats.norm.cdf(-d2) - S * stats.norm.cdf(-d1)

# Parameters
S0, K, T, r, sigma = 100, 100, 1.0, 0.05, 0.20
bs_call_price = bs_call(S0, K, T, r, sigma)
bs_put_price = bs_put(S0, K, T, r, sigma)
print(f"BS Call: ${bs_call_price:.4f}")
print(f"BS Put:  ${bs_put_price:.4f}")

These are our reference prices. We are pricing an at-the-money option ($S_0 = K = 100$) with one year
to expiry, 5% risk-free rate, and 20% volatility. Notice that the call is slightly more expensive
than the put -- this is because of the risk-neutral drift: the stock is expected to grow at rate $r$
under the pricing measure, making upside (call) more likely than downside (put).

Every finite difference scheme in this notebook should produce prices very close to these values.
The difference is the **discretization error**, which shrinks as we refine the grid.> **Key Concept:** These analytical Black-Scholes prices serve as our "ground truth." Every finite difference scheme we build should converge to these values as we refine the grid. If a scheme gives a very different answer, either the grid is too coarse or there's a bug.


---

## 4. Grid Setup -- A Spreadsheet of Stock Prices and Times

Before we can solve the PDE numerically, we need to **discretize** the continuous domain. This means
replacing the smooth $(S, t)$ plane with a finite set of grid points.

### The Two Dimensions

We discretize along two axes:

1. **Stock price** $S$: from $0$ to some large upper bound $S_{\max}$, divided into $M$ equal
   intervals. This gives $M + 1$ grid points with spacing $\Delta S = S_{\max} / M$.

2. **Time** $t$: from $0$ (today) to $T$ (expiry), divided into $N$ equal intervals. This gives
   $N + 1$ grid points with spacing $\Delta t = T / N$.

We denote the option value at grid point $(j, n)$ as:

$$V_j^n \approx V(j \cdot \Delta S, \; n \cdot \Delta t)$$

where $j = 0, 1, \ldots, M$ indexes the stock price and $n = 0, 1, \ldots, N$ indexes the time.

> **Key Concept:** Think of the grid as a **spreadsheet**. Each row corresponds to a different stock
> price. Each column corresponds to a different point in time. The last column (time $T$) is easy --
> it is just the payoff function. Our job is to fill in the rest of the spreadsheet, working from
> right to left (backward in time).

### Choosing Grid Parameters

Three choices matter:

- **$S_{\max}$**: Must be large enough that the boundary condition at $S = S_{\max}$ does not pollute
  the solution near $S_0$. A common rule of thumb is $S_{\max} = 3K$ to $5K$. We use $S_{\max} = 300$
  (three times the strike).

- **$M$ (spatial steps)**: More steps means finer resolution in the stock price dimension. Typical
  values range from 100 to 1000. We start with $M = 200$.

- **$N$ (time steps)**: More steps means finer resolution in time. For the explicit method, $N$
  must be very large to satisfy the stability condition. For implicit and Crank-Nicolson, $N$ can
  be much smaller. We start with $N = 1000$.

### Boundary Conditions

At the edges of the grid, we need boundary conditions:

- **At $S = 0$**: A call is worthless ($V = 0$). A put is worth the present value of the strike
  ($V = K e^{-r(T-t)}$).
- **At $S = S_{\max}$**: A call is approximately $S_{\max} - K e^{-r(T-t)}$ (deep in the money).
  A put is worthless ($V = 0$).
- **At $t = T$**: The option value equals the payoff: $V(S, T) = \phi(S)$.

The next cell creates the grid and prints its dimensions. This `create_grid` function will be used
by all three finite difference methods.

In [ ]:
def create_grid(S_max, M, T, N):
    """Create finite difference grid."""
    dS = S_max / M
    dt = T / N
    S_grid = np.linspace(0, S_max, M + 1)
    t_grid = np.linspace(0, T, N + 1)
    return S_grid, t_grid, dS, dt

S_max = 300  # sufficiently large
M = 200      # stock price steps
N = 1000     # time steps

S_grid, t_grid, dS, dt = create_grid(S_max, M, T, N)
print(f"Grid: {M+1} × {N+1}, ΔS = {dS:.2f}, Δt = {dt:.6f}")

Our grid has 201 stock prices (from $0 to $300 in $1.50 increments) and 1001 time points (from
$t = 0$ to $t = 1$ in steps of $0.001$). That is about 201,000 grid points in total -- a modest
size that any laptop can handle.

Notice that $\Delta S = 1.50$: the grid does not land exactly on $S_0 = 100$. The closest grid point
is at $S = 100.50$. For higher accuracy near $S_0$, we could use a non-uniform grid (finer near the
strike, coarser far away), but uniform grids are simpler and sufficient for our purposes.> **Important:** The choice of $S_{\max}$ matters. If it's too small, the boundary conditions contaminate the interior solution. If it's too large, the grid is wasteful. A rule of thumb: $S_{\max} \approx 3K$ to $4K$ works well for most options.

> **Common Mistake:** Students sometimes confuse the number of spatial steps ($M$) with the number of time steps ($N$). Both affect accuracy, but they play different roles — $M$ controls the resolution in stock price, $N$ controls resolution in time. The stability of the explicit method depends on the *ratio* $\Delta t / (\Delta S)^2$.


---

## 5. Explicit Finite Difference -- Simple but Fragile

The **explicit method** (also called the **forward Euler** scheme in the spatial direction) is the
simplest way to step backward through time. The idea: at each time step, compute the new option
values *directly* from the values at the next time step, using only multiplication and addition.
No equation solving is needed.

### Deriving the Scheme

We approximate each derivative in the Black-Scholes PDE using finite differences. At grid point
$(j, n+1)$:

- **Time derivative** (backward difference):
  $$\frac{\partial V}{\partial t} \approx \frac{V_j^{n+1} - V_j^n}{\Delta t}$$

- **First spatial derivative** (central difference):
  $$\frac{\partial V}{\partial S} \approx \frac{V_{j+1}^{n+1} - V_{j-1}^{n+1}}{2\Delta S}$$

- **Second spatial derivative** (central difference):
  $$\frac{\partial^2 V}{\partial S^2} \approx \frac{V_{j+1}^{n+1} - 2V_j^{n+1} + V_{j-1}^{n+1}}{(\Delta S)^2}$$

Substituting into the PDE with $S_j = j \cdot \Delta S$ and rearranging, we get:

$$V_j^n = a_j \, V_{j-1}^{n+1} + b_j \, V_j^{n+1} + c_j \, V_{j+1}^{n+1}$$

where the coefficients are:

$$a_j = \frac{\Delta t}{2}(\sigma^2 j^2 - r j), \quad
  b_j = 1 - \Delta t(\sigma^2 j^2 + r), \quad
  c_j = \frac{\Delta t}{2}(\sigma^2 j^2 + r j)$$

### Why "Explicit"?

The formula above tells us that $V_j^n$ (the value at the *current* time step) depends only on
$V_{j-1}^{n+1}$, $V_j^{n+1}$, and $V_{j+1}^{n+1}$ (values at the *next* time step, which we
already know). We can compute each value independently -- no system of equations to solve. This
is what makes it "explicit".

> **Key Concept:** "Explicit" means we can compute each new value directly from the previous time
> step. Think of filling in a spreadsheet row by row, where each cell is a simple weighted average
> of three cells in the row to the right. Fast and simple -- but there is a catch.

### The Catch: It Can Blow Up

The explicit method has a **stability condition** (the CFL condition -- named after Courant, Friedrichs,
and Lewy). The coefficients $a_j$, $b_j$, $c_j$ must all be non-negative for the scheme to behave
well. The binding constraint is on $b_j$:

$$b_j = 1 - \Delta t(\sigma^2 j^2 + r) \geq 0 \quad \Longrightarrow \quad \Delta t \leq \frac{1}{\sigma^2 j^2 + r}$$

Since this must hold for all $j$ up to $M$, the worst case is $j = M$. With $M = 200$ and
$\sigma = 0.20$, we need $\Delta t \leq 1/(0.04 \cdot 40000 + 0.05) \approx 0.000625$. For
$T = 1$, this means at least $N = 1600$ time steps. If we use fewer, the solution **oscillates
wildly and eventually overflows to infinity**.

> **Important:** The CFL condition is the Achilles' heel of the explicit method. It forces you to take
> very small time steps, which makes the method slow for fine spatial grids. If you double $M$
> (the number of stock price points), you must *quadruple* $N$ (the number of time steps) to maintain
> stability. This $O(M^2)$ scaling is why practitioners prefer implicit or Crank-Nicolson methods.

The next cell implements the explicit scheme. We initialize the grid with the payoff at expiry
(the rightmost column of our spreadsheet), then loop backward through time, filling in each column
using the explicit formula.

In [ ]:
def explicit_fd(S_max, M, N, K, T, r, sigma, option_type='call'):
    """Explicit finite difference for European options."""
    S_grid, t_grid, dS, dt = create_grid(S_max, M, T, N)
    
    # Initialize with payoff at t=T
    V = np.zeros((M + 1, N + 1))
    if option_type == 'call':
        V[:, N] = np.maximum(S_grid - K, 0)
    else:
        V[:, N] = np.maximum(K - S_grid, 0)
    
    # Coefficients
    j = np.arange(0, M + 1)
    a = 0.5 * dt * (sigma**2 * j**2 - r * j)
    b = 1 - dt * (sigma**2 * j**2 + r)
    c = 0.5 * dt * (sigma**2 * j**2 + r * j)
    
    # Backward time-stepping
    for n in range(N - 1, -1, -1):
        for jj in range(1, M):
            V[jj, n] = a[jj] * V[jj-1, n+1] + b[jj] * V[jj, n+1] + c[jj] * V[jj+1, n+1]
        
        # Boundary conditions
        if option_type == 'call':
            V[0, n] = 0
            V[M, n] = S_max - K * np.exp(-r * (T - t_grid[n]))
        else:
            V[0, n] = K * np.exp(-r * (T - t_grid[n]))
            V[M, n] = 0
    
    return S_grid, V

S_grid_ex, V_ex = explicit_fd(S_max, M, N, K, T, r, sigma, 'call')

# Find price at S0
idx_S0 = np.argmin(np.abs(S_grid_ex - S0))
fd_call = V_ex[idx_S0, 0]
print(f"Explicit FD call: ${fd_call:.4f}")
print(f"BS call:          ${bs_call_price:.4f}")
print(f"Error:            ${abs(fd_call - bs_call_price):.6f}")

### Interpreting the Explicit Method Results

The explicit scheme produces a call price very close to the Black-Scholes analytical value. The
error is small because we used $N = 1000$ time steps -- well above the stability threshold.

But what happens if we use too few time steps? The explicit method does not degrade gracefully --
it does not just become less accurate. It **completely blows up**, producing wildly oscillating or
infinite values. This is the stability issue in action.

> **Practical Rule of Thumb:** For the explicit method, use at least $N \geq \frac{M^2 \sigma^2 S_{\max}^2}{2 S_{\max}^2} = \frac{M^2 \sigma^2}{2}$
> time steps. For $M = 200$ and $\sigma = 0.20$, that is at least $N \geq 4000$. In practice, the
> implicit and Crank-Nicolson methods avoid this issue entirely.
> **Important:** The explicit method is conceptually simple and easy to implement, making it a good teaching tool. However, its conditional stability makes it impractical for production use. In practice, Crank-Nicolson is almost always preferred.


---

## 6. Stability Analysis -- When the Explicit Method Blows Up

Let us see the CFL instability in action. We will run the explicit method twice: once with enough
time steps (stable), and once with too few (unstable). The difference is dramatic.

### The CFL Condition in Detail

The **Courant-Friedrichs-Lewy (CFL) condition** is one of the most important results in numerical
analysis. For the explicit scheme applied to the Black-Scholes PDE, stability requires:

$$\Delta t \leq \frac{1}{\sigma^2 j_{\max}^2 + r}$$

Think of it this way: the explicit method computes each new value as a weighted average of three
neighboring values at the next time step. For this to make sense, the weights must all be positive.
If $\Delta t$ is too large, some weights become negative, and the "average" can exceed the range of
the inputs -- the solution overshoots, then overcorrects, then overshoots more, spiraling out of
control.

> **Key Concept:** Think of stability like a ball on a hilltop. A stable method is like a ball in a
> valley -- small perturbations die out and the solution converges. An unstable method is like a ball
> on a hilltop -- even tiny numerical rounding errors get amplified at every step, until the solution
> is meaningless.

The next cell demonstrates this by running the explicit method with $N = 1000$ (stable) and
$N = 50$ (violently unstable).

In [ ]:
# Demonstrate instability
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, N_test, title in [(axes[0], 1000, 'Stable (N=1000)'), 
                            (axes[1], 50, 'Unstable (N=50)')]:
    try:
        S_g, V_test = explicit_fd(S_max, M, N_test, K, T, r, sigma, 'call')
        ax.plot(S_g, V_test[:, 0], color=PRIMARY, linewidth=2, label='FD')
        ax.plot(S_g, [bs_call(s, K, T, r, sigma) for s in S_g], '--', 
                color=SECONDARY, linewidth=2, label='BS analytical')
        ax.set_xlim(50, 150)
        ax.set_ylim(-5, 60)
    except:
        ax.text(0.5, 0.5, 'Overflow', transform=ax.transAxes, ha='center', fontsize=20)
    
    dt_test = T / N_test
    j_max_stable = int(np.sqrt(1 / (sigma**2 * dt_test) - r / sigma**2))
    ax.set_xlabel('Stock Price')
    ax.set_ylabel('Option Price')
    ax.set_title(f'{title}, Δt={dt_test:.5f}')
    ax.legend()

plt.tight_layout()
plt.show()

### Interpreting the Stability Plots

The left panel shows the stable case ($N = 1000$): the finite difference solution (blue) closely
tracks the Black-Scholes analytical solution (dashed orange). Everything looks clean.

The right panel shows what happens when we violate the CFL condition ($N = 50$): the solution
oscillates wildly, producing negative option prices and values that bear no resemblance to the
true answer. In extreme cases, the values overflow to infinity.

This is not a subtle inaccuracy -- it is a **catastrophic failure**. The explicit method gives no
warning that something is wrong. It just produces garbage. This is why understanding stability
conditions is essential, and why practitioners often prefer unconditionally stable methods.

> **Important:** In a production system, you would never want a pricing engine that can silently
> produce garbage. This is the main practical argument for implicit and Crank-Nicolson methods:
> they are **unconditionally stable**, meaning they produce reasonable (if possibly inaccurate)
> results for any choice of time step.> **Key Concept:** The CFL (Courant-Friedrichs-Lewy) condition for the explicit method is:

$$\Delta t \leq \frac{1}{\sigma^2 j_{\max}^2 + r j_{\max}}$$

where $j_{\max} = M$ is the maximum grid index. In practice, this means the number of time steps $N$ must grow as $M^2$ — making the explicit method very expensive for fine spatial grids. The implicit and Crank-Nicolson methods have no such restriction.


---

## 7. Implicit Finite Difference -- Always Stable

The **implicit method** (also called **backward Euler**) takes a different approach. Instead of
evaluating the spatial derivatives at the *known* time step $n+1$, it evaluates them at the
*unknown* time step $n$. This seemingly small change has a profound consequence: the method
becomes **unconditionally stable**.

### Deriving the Scheme

We use the same spatial finite differences as before, but now evaluate them at time $n$:

$$\frac{V_j^{n+1} - V_j^n}{\Delta t} + \frac{1}{2}\sigma^2 S_j^2 \frac{V_{j+1}^n - 2V_j^n + V_{j-1}^n}{(\Delta S)^2} + rS_j \frac{V_{j+1}^n - V_{j-1}^n}{2\Delta S} - rV_j^n = 0$$

Rearranging, this becomes:

$$-\alpha_j \, V_{j-1}^n + \beta_j \, V_j^n - \gamma_j \, V_{j+1}^n = V_j^{n+1}$$

where:

$$\alpha_j = \frac{\Delta t}{2}(\sigma^2 j^2 - rj), \quad
  \beta_j = 1 + \Delta t(\sigma^2 j^2 + r), \quad
  \gamma_j = \frac{\Delta t}{2}(\sigma^2 j^2 + rj)$$

### Why This Leads to a System of Equations

Notice the key difference from the explicit method: the unknown values $V_{j-1}^n$, $V_j^n$, and
$V_{j+1}^n$ all appear on the **left-hand side**. They are coupled -- you cannot compute one without
knowing the others. This means we must solve a **system of linear equations** at each time step.

Written in matrix form for all interior points $j = 1, \ldots, M-1$:

$$\mathbf{A} \, \mathbf{V}^n = \mathbf{V}^{n+1} + \text{boundary terms}$$

where $\mathbf{A}$ is a **tridiagonal matrix** -- it has nonzero entries only on the main diagonal
and the two adjacent diagonals.

> **Key Concept:** "Implicit" means the new values are all coupled together and must be solved
> simultaneously as a system of equations. It is more work per time step than the explicit method,
> but the payoff is enormous: you can take *any* size time step and the solution will not blow up.

### The Thomas Algorithm for Tridiagonal Systems

Normally, solving a system of $n$ equations takes $O(n^3)$ operations (Gaussian elimination). But
our matrix is **tridiagonal** -- each equation involves only three unknowns. This special structure
allows us to use the **Thomas algorithm**, a streamlined version of Gaussian elimination that runs
in just $O(n)$ operations.

The Thomas algorithm works in two passes:

1. **Forward sweep**: Eliminate the sub-diagonal entries, transforming the system into upper
   triangular form.
2. **Back substitution**: Solve for each unknown starting from the last equation and working
   backward.

This makes the implicit method nearly as fast per time step as the explicit method -- and since
we can use far fewer time steps (no stability constraint), it is actually faster overall.

> **Important:** The Thomas algorithm is specific to tridiagonal systems. If we had a
> two-dimensional PDE (e.g., pricing a two-asset option), the matrix would no longer be tridiagonal,
> and we would need more sophisticated solvers. But for one-dimensional problems like Black-Scholes,
> the Thomas algorithm is ideal.

The next cell implements both the Thomas algorithm and the full implicit finite difference scheme.
Notice that we use only $N = 200$ time steps -- far fewer than the explicit method needed -- and
the solution is perfectly stable.

In [ ]:
def thomas_algorithm(a, b, c, d):
    """Solve tridiagonal system Ax=d using Thomas algorithm.
    a: sub-diagonal, b: main diagonal, c: super-diagonal, d: RHS.
    """
    n = len(d)
    c_ = np.zeros(n)
    d_ = np.zeros(n)
    x = np.zeros(n)
    
    # Forward sweep
    c_[0] = c[0] / b[0]
    d_[0] = d[0] / b[0]
    for i in range(1, n):
        m = b[i] - a[i] * c_[i-1]
        c_[i] = c[i] / m if i < n - 1 else 0
        d_[i] = (d[i] - a[i] * d_[i-1]) / m
    
    # Back substitution
    x[-1] = d_[-1]
    for i in range(n - 2, -1, -1):
        x[i] = d_[i] - c_[i] * x[i+1]
    
    return x

def implicit_fd(S_max, M, N, K, T, r, sigma, option_type='call'):
    """Implicit finite difference for European options."""
    S_grid, t_grid, dS, dt = create_grid(S_max, M, T, N)
    
    V = np.zeros(M + 1)
    if option_type == 'call':
        V = np.maximum(S_grid - K, 0)
    else:
        V = np.maximum(K - S_grid, 0)
    
    # Interior points: j = 1, ..., M-1
    j = np.arange(1, M)
    alpha = 0.5 * dt * (sigma**2 * j**2 - r * j)
    beta = 1 + dt * (sigma**2 * j**2 + r)
    gamma = 0.5 * dt * (sigma**2 * j**2 + r * j)
    
    V_full = np.zeros((M + 1, N + 1))
    V_full[:, N] = V.copy()
    
    for n in range(N - 1, -1, -1):
        # RHS
        rhs = V[1:M].copy()
        
        # Boundary conditions
        if option_type == 'call':
            V_lower = 0
            V_upper = S_max - K * np.exp(-r * (T - t_grid[n]))
        else:
            V_lower = K * np.exp(-r * (T - t_grid[n]))
            V_upper = 0
        
        rhs[0] += alpha[0] * V_lower
        rhs[-1] += gamma[-1] * V_upper
        
        # Solve tridiagonal system
        sub = -alpha[1:]     # sub-diagonal
        main = beta          # main diagonal
        sup = -gamma[:-1]    # super-diagonal
        
        V[1:M] = thomas_algorithm(np.concatenate([[0], sub]), main, np.concatenate([sup, [0]]), rhs)
        V[0] = V_lower
        V[M] = V_upper
        V_full[:, n] = V.copy()
    
    return S_grid, V_full

S_grid_im, V_im = implicit_fd(S_max, M, 200, K, T, r, sigma, 'call')  # works even with few time steps
idx_S0 = np.argmin(np.abs(S_grid_im - S0))
print(f"Implicit FD (N=200): ${V_im[idx_S0, 0]:.4f}")
print(f"BS analytical:       ${bs_call_price:.4f}")

### Interpreting the Implicit Method Results

The implicit method produces a good approximation with only $N = 200$ time steps -- five times fewer
than we used for the explicit method. And if we had used even fewer (say $N = 10$), the answer would
be inaccurate but still *finite and reasonable*. No blow-up, no oscillations, no garbage.

The trade-off: the implicit method is only **first-order accurate** in time, meaning the error
decreases proportionally to $\Delta t$. Halving the time step halves the error. This is decent
but not great -- the Crank-Nicolson method achieves second-order accuracy (halving the time step
quarters the error) at essentially the same computational cost.

> **Key Concept:** The implicit method's unconditional stability comes from the fact that the
> tridiagonal matrix $\mathbf{A}$ has all positive diagonal entries and is diagonally dominant.
> This means the system $\mathbf{A}\mathbf{V}^n = \mathbf{d}$ always has a unique solution, and
> small perturbations in $\mathbf{d}$ produce small changes in $\mathbf{V}^n$. Errors cannot grow.> **Key Concept:** The Thomas algorithm solves the tridiagonal system in $O(M)$ operations — far cheaper than general matrix inversion ($O(M^3)$). This efficiency is what makes implicit and Crank-Nicolson methods practical. Without it, we'd need to invert a 200×200 matrix at every time step.

> **Common Mistake:** When implementing the Thomas algorithm, be careful about boundary conditions. The first and last rows of the tridiagonal system encode the boundary conditions ($V = 0$ at $S = 0$ for a call, and $V = S - Ke^{-r(T-t)}$ at $S_{\max}$). Getting these wrong shifts the entire solution.


---

## 8. Crank-Nicolson -- The Best of Both Worlds

The **Crank-Nicolson** scheme is the gold standard for parabolic PDEs. It combines the unconditional
stability of the implicit method with second-order accuracy in time. The idea is beautifully simple:
**average the explicit and implicit schemes**.

### Deriving the Scheme

Instead of evaluating the spatial derivatives at time $n+1$ (explicit) or time $n$ (implicit),
Crank-Nicolson evaluates them at the midpoint $n + 1/2$ by taking the average:

$$\frac{V_j^{n+1} - V_j^n}{\Delta t} = \frac{1}{2}\bigl[\mathcal{L}V_j^{n+1} + \mathcal{L}V_j^n\bigr]$$

where $\mathcal{L}$ is the spatial differential operator from the Black-Scholes PDE:

$$\mathcal{L}V = \frac{1}{2}\sigma^2 S^2 \frac{\partial^2 V}{\partial S^2} + rS\frac{\partial V}{\partial S} - rV$$

This gives a system of the form:

$$(\mathbf{I} - \tfrac{1}{2}\Delta t \, \mathbf{L}) \, \mathbf{V}^n = (\mathbf{I} + \tfrac{1}{2}\Delta t \, \mathbf{L}) \, \mathbf{V}^{n+1} + \text{boundary terms}$$

The right-hand side uses known values (from the next time step), and the left-hand side is a
tridiagonal system that we solve with the Thomas algorithm -- exactly as in the implicit method.

### Why Averaging Gives Second-Order Accuracy

The explicit method evaluates the spatial operator at time $n+1$ and has error $O(\Delta t)$. The
implicit method evaluates it at time $n$ and also has error $O(\Delta t)$. But these two errors
have **opposite signs**: one overestimates, the other underestimates. When we average them, the
first-order errors cancel, leaving an error of $O(\Delta t^2)$.

This is the same principle behind the **trapezoidal rule** for numerical integration: approximating
an integral with trapezoids (averaging the left and right endpoint values) is more accurate than
using just one endpoint.

> **Key Concept:** Crank-Nicolson achieves second-order accuracy by "time-centering" the spatial
> derivatives at the midpoint between two time steps. Half the weight goes to the known time step,
> half to the unknown. This cancels the leading error term, giving much better accuracy for
> essentially the same computational effort as the implicit method.

### Detailed Comparison of the Three Methods

Here is the complete comparison:

| Property | Explicit | Implicit | Crank-Nicolson |
|:---------|:---------|:---------|:---------------|
| **Accuracy in time** | $O(\Delta t)$ (first-order) | $O(\Delta t)$ (first-order) | $O(\Delta t^2)$ (second-order) |
| **Accuracy in space** | $O(\Delta S^2)$ (second-order) | $O(\Delta S^2)$ (second-order) | $O(\Delta S^2)$ (second-order) |
| **Stability** | Conditional (CFL) | Unconditional | Unconditional |
| **Work per step** | $O(M)$ -- no solve | $O(M)$ -- Thomas algorithm | $O(M)$ -- Thomas algorithm |
| **System to solve** | None | Tridiagonal | Tridiagonal |
| **Total work for error $\epsilon$** | $O(1/\epsilon^{3/2})$ | $O(1/\epsilon^{3/2})$ | $O(1/\epsilon)$ |
| **Industry use** | Rare (stability issues) | Sometimes (robust but slow to converge) | Standard (stable + accurate) |

Crank-Nicolson is the clear winner: same cost per step as implicit, but the error decreases
quadratically rather than linearly with the time step. This is why it is the **industry standard**
for one-dimensional PDE problems in finance.

> **Important:** Crank-Nicolson has one known issue: it can produce small **oscillations** near
> discontinuities in the payoff (such as the kink at the strike for a call or put). This is because
> second-order methods struggle with non-smooth data. A practical fix is **Rannacher smoothing**:
> use two or three fully implicit steps at the start (near the payoff), then switch to Crank-Nicolson
> for the remaining steps. This eliminates the oscillations while preserving the overall second-order
> convergence.

The next cell implements the Crank-Nicolson scheme. Note the structure: the right-hand side applies
the "explicit half" to the known values, and the left-hand side solves the "implicit half" with
the Thomas algorithm.

In [ ]:
def crank_nicolson_fd(S_max, M, N, K, T, r, sigma, option_type='call'):
    """Crank-Nicolson finite difference for European options."""
    S_grid, t_grid, dS, dt = create_grid(S_max, M, T, N)
    
    V = np.zeros(M + 1)
    if option_type == 'call':
        V = np.maximum(S_grid - K, 0)
    else:
        V = np.maximum(K - S_grid, 0)
    
    j = np.arange(1, M)
    alpha = 0.25 * dt * (sigma**2 * j**2 - r * j)
    beta_diag = -0.5 * dt * (sigma**2 * j**2 + r)
    gamma = 0.25 * dt * (sigma**2 * j**2 + r * j)
    
    V_full = np.zeros((M + 1, N + 1))
    V_full[:, N] = V.copy()
    
    for n in range(N - 1, -1, -1):
        # Boundary values at time n
        if option_type == 'call':
            V_lower = 0
            V_upper = S_max - K * np.exp(-r * (T - t_grid[n]))
            V_lower_next = 0
            V_upper_next = S_max - K * np.exp(-r * (T - t_grid[n+1]))
        else:
            V_lower = K * np.exp(-r * (T - t_grid[n]))
            V_upper = 0
            V_lower_next = K * np.exp(-r * (T - t_grid[n+1]))
            V_upper_next = 0
        
        # RHS: (I + B) V^{n+1} where B has the explicit part
        rhs = np.zeros(M - 1)
        for jj_idx in range(M - 1):
            jj = jj_idx + 1
            rhs[jj_idx] = -alpha[jj_idx] * V[jj-1] + (1 + beta_diag[jj_idx]) * V[jj] - gamma[jj_idx] * V[jj+1]
        
        rhs[0] += alpha[0] * V_lower + alpha[0] * V_lower_next
        rhs[-1] += gamma[-1] * V_upper + gamma[-1] * V_upper_next
        
        # LHS: (I - B) V^n
        sub = alpha[1:]          # sub-diagonal
        main = 1 - beta_diag     # main diagonal
        sup = gamma[:-1]         # super-diagonal
        
        V[1:M] = thomas_algorithm(
            np.concatenate([[0], sub]),
            main,
            np.concatenate([sup, [0]]),
            rhs
        )
        V[0] = V_lower
        V[M] = V_upper
        V_full[:, n] = V.copy()
    
    return S_grid, V_full

S_grid_cn, V_cn = crank_nicolson_fd(S_max, M, 200, K, T, r, sigma, 'call')
idx_S0 = np.argmin(np.abs(S_grid_cn - S0))
print(f"Crank-Nicolson (N=200): ${V_cn[idx_S0, 0]:.4f}")
print(f"BS analytical:          ${bs_call_price:.4f}")

### Interpreting the Crank-Nicolson Results

With only $N = 200$ time steps, Crank-Nicolson produces a price that matches the analytical
Black-Scholes value to several decimal places. Compare this to the implicit method at the same
resolution: Crank-Nicolson is noticeably more accurate because it converges at rate $O(\Delta t^2)$
rather than $O(\Delta t)$.

And unlike the explicit method, there is no stability concern. We could use $N = 10$ and still
get a finite, sensible (if somewhat inaccurate) answer.This is why Crank-Nicolson is the industry standard:
- Same computational cost per step as the implicit method (one tridiagonal solve)
- But **second-order accurate** in time ($O(\Delta t^2)$) vs first-order ($O(\Delta t)$) for explicit/implicit
- Unconditionally stable, so you can use whatever time step you like

> **CFA Exam Tip:** While finite difference methods aren't directly tested on the CFA exam, understanding that numerical methods exist for pricing complex derivatives (American options, barriers) is conceptually important.


---

## 9. European Option Pricing -- Visual Comparison

Now let us put all three methods side by side. The plot below shows the option price as a function
of the stock price for each method, overlaid with the Black-Scholes analytical curve.

If the methods are working correctly, all curves should be nearly indistinguishable. Any visible
deviation indicates discretization error.

Note the grid resolutions used:
- **Explicit:** $M = 200$, $N = 1000$ (needs many time steps for stability)
- **Implicit:** $M = 200$, $N = 200$ (stable with fewer steps, but first-order)
- **Crank-Nicolson:** $M = 200$, $N = 200$ (stable and second-order)

In [ ]:
# Compare all three methods
fig, ax = plt.subplots()

S_bs = S_grid_cn
V_bs = np.array([bs_call(s, K, T, r, sigma) for s in S_bs])

ax.plot(S_bs, V_bs, 'k-', linewidth=3, label='BS analytical', alpha=0.5)
ax.plot(S_grid_ex, V_ex[:, 0], '--', color=PRIMARY, linewidth=2, label='Explicit')
ax.plot(S_grid_im, V_im[:, 0], '--', color=SECONDARY, linewidth=2, label='Implicit')
ax.plot(S_grid_cn, V_cn[:, 0], '--', color=TERTIARY, linewidth=2, label='Crank-Nicolson')

ax.set_xlim(60, 140)
ax.set_ylim(0, 45)
ax.set_xlabel('Stock Price ($)')
ax.set_ylabel('Call Price ($)')
ax.set_title('European Call — FD Methods vs Analytical')
ax.legend()
plt.tight_layout()
plt.show()

### Interpreting the Comparison Plot

All three finite difference methods produce curves that are visually indistinguishable from the
Black-Scholes analytical solution. This confirms that our implementations are correct.

The differences between the methods become apparent only when we zoom in very closely or look at
the numerical errors. The convergence analysis later in this notebook will quantify exactly how
fast each method converges to the true answer as we refine the grid.

> **Key Concept:** For European vanilla options, all three methods give good results with a
> reasonable grid. The differences matter most for exotic options, very fine accuracy requirements,
> or computational efficiency at scale.The tiny differences between the methods only become visible when you zoom in near the strike price ($S = K$), where the payoff has a kink. This kink causes the most numerical difficulty:

- **Explicit** may show small oscillations near the kink (if close to the stability limit)
- **Implicit** smooths the kink aggressively (numerical diffusion)
- **Crank-Nicolson** handles it best but can show tiny oscillations near $T = 0$

> **Important:** In production systems, practitioners often use **Rannacher time-stepping** — a few initial implicit steps followed by Crank-Nicolson. The implicit start smooths the payoff kink, eliminating the oscillation problem while preserving CN's second-order accuracy for the rest of the grid.


---

## 10. American Options -- The Free Boundary Problem

Everything so far has dealt with **European** options, which can only be exercised at expiry. Now
we tackle the much harder problem of **American** options, which can be exercised at any time.

### The "Exercise or Hold?" Decision

At every moment before expiry, the holder of an American option faces a choice:

- **Exercise now** and receive the intrinsic value $\max(K - S, 0)$ (for a put)
- **Hold** and keep the option alive, hoping for a better outcome later

The option value must satisfy:

$$V(S, t) \geq \phi(S) \quad \text{for all } S, t$$

where $\phi(S)$ is the payoff (intrinsic value). If the option were ever worth less than its
intrinsic value, the holder would exercise immediately and pocket the difference. So the option
price must always be at least as large as the exercise value.

### The Free Boundary

For an American put, there exists a critical stock price $S^*(t)$ -- called the **early exercise
boundary** or **free boundary** -- that separates two regions:

- **Below $S^*(t)$**: The option should be exercised immediately. Its value equals the intrinsic
  value: $V = K - S$.
- **Above $S^*(t)$**: The option should be held. Its value satisfies the Black-Scholes PDE and
  exceeds the intrinsic value.

The boundary $S^*(t)$ is not known in advance -- it must be found as part of the solution. This
is what makes the problem a "free boundary" problem, and it is the reason there is no known
closed-form solution for the American put.

> **Key Concept:** The American option problem is a **linear complementarity problem (LCP)**: at
> each grid point, either the PDE holds with equality (the option is alive), or the option value
> equals the intrinsic value (the option is exercised). Both cannot happen simultaneously (except
> at the boundary itself), and at least one must hold everywhere.

### Projected Successive Over-Relaxation (PSOR)

To handle the early exercise constraint within our finite difference framework, we use the
**Projected SOR (PSOR)** method. This is an iterative solver that:

1. Performs a standard SOR update (an iterative method for solving linear systems, faster than
   simple Gauss-Seidel due to the over-relaxation parameter $\omega > 1$).
2. **Projects** the result: if the computed value falls below the intrinsic value, snap it back
   up to the intrinsic value.

The projection step is what enforces the early exercise constraint. The algorithm converges to
the correct American option price, automatically finding the free boundary.

We use the Crank-Nicolson scheme for the PDE part (for its superior accuracy) combined with
PSOR for the constraint enforcement.

The next cell implements the full American put pricer and reports the early exercise premium --
the amount by which the American put exceeds the European put.

In [ ]:
def american_put_cn_psor(S_max, M, N, K, T, r, sigma, omega=1.2, tol=1e-8, max_iter=1000):
    """American put via Crank-Nicolson with PSOR for the free boundary."""
    S_grid, t_grid, dS, dt = create_grid(S_max, M, T, N)
    payoff = np.maximum(K - S_grid, 0)
    
    V = payoff.copy()
    
    j = np.arange(1, M)
    alpha = 0.25 * dt * (sigma**2 * j**2 - r * j)
    beta_diag = -0.5 * dt * (sigma**2 * j**2 + r)
    gamma = 0.25 * dt * (sigma**2 * j**2 + r * j)
    
    early_exercise = np.zeros((M + 1, N + 1), dtype=bool)
    V_full = np.zeros((M + 1, N + 1))
    V_full[:, N] = V.copy()
    
    for n in range(N - 1, -1, -1):
        V_lower = K * np.exp(-r * (T - t_grid[n]))
        V_upper = 0
        
        # RHS
        rhs = np.zeros(M - 1)
        for jj_idx in range(M - 1):
            jj = jj_idx + 1
            rhs[jj_idx] = -alpha[jj_idx] * V[jj-1] + (1 + beta_diag[jj_idx]) * V[jj] - gamma[jj_idx] * V[jj+1]
        rhs[0] += alpha[0] * (V_lower + V[0])
        rhs[-1] += gamma[-1] * (V_upper + V[M])
        
        # PSOR iteration
        main = 1 - beta_diag
        V_new = V[1:M].copy()
        
        for it in range(max_iter):
            V_old = V_new.copy()
            for jj_idx in range(M - 1):
                # Gauss-Seidel update
                residual = rhs[jj_idx]
                if jj_idx > 0:
                    residual -= alpha[jj_idx] * V_new[jj_idx - 1]
                if jj_idx < M - 2:
                    residual -= gamma[jj_idx] * V_new[jj_idx + 1]
                
                gs_val = residual / main[jj_idx]
                # SOR with projection
                V_new[jj_idx] = max(
                    payoff[jj_idx + 1],
                    V_new[jj_idx] + omega * (gs_val - V_new[jj_idx])
                )
            
            if np.max(np.abs(V_new - V_old)) < tol:
                break
        
        V[1:M] = V_new
        V[0] = V_lower
        V[M] = V_upper
        V_full[:, n] = V.copy()
        early_exercise[1:M, n] = (V[1:M] <= payoff[1:M] + 1e-10) & (payoff[1:M] > 0)
    
    return S_grid, V_full, early_exercise

S_am, V_am, ee = american_put_cn_psor(S_max, M, 500, K, T, r, sigma)
idx_S0 = np.argmin(np.abs(S_am - S0))
eur_put = bs_put(S0, K, T, r, sigma)
am_put = V_am[idx_S0, 0]

print(f"European put (BS): ${eur_put:.4f}")
print(f"American put (FD): ${am_put:.4f}")
print(f"Early exercise premium: ${am_put - eur_put:.4f}")

### Interpreting the American Put Price

The American put is more expensive than the European put. The difference is the **early exercise
premium** -- the value of having the *option* to exercise early.

Why does early exercise matter for puts? Consider a put that is deep in the money ($S \ll K$).
The intrinsic value is $K - S$, which could be very large. If you hold the European put, you
must wait until expiry to receive this payoff, and in the meantime the stock might recover.
With the American put, you can exercise now, invest the proceeds at the risk-free rate, and
earn interest. When the interest earned exceeds the remaining time value, early exercise is optimal.

> **Key Concept:** The early exercise premium is always non-negative ($V_{\text{American}} \geq
> V_{\text{European}}$), because having more choices can never make you worse off. For calls on
> non-dividend-paying stocks, the premium is actually zero -- it is never optimal to exercise a
> call early. But for puts, the premium is typically positive and can be significant.

The next cell visualizes the American put price curve alongside the European put and the intrinsic
value, highlighting the early exercise premium.The early exercise premium comes from the right to exercise when the put is deep in the money. At very low stock prices, holding the put gives you essentially no more optionality — you're better off exercising immediately and investing the proceeds at the risk-free rate.

> **Key Concept:** The **exercise boundary** $S^*(t)$ divides the grid into two regions:
> - $S > S^*(t)$: "Continuation region" — hold the option
> - $S \leq S^*(t)$: "Exercise region" — exercise immediately
>
> The PSOR algorithm finds this boundary automatically as a byproduct of solving the constrained system.


In [ ]:
# Plot American vs European put
fig, ax = plt.subplots()
V_eur_put = np.array([bs_put(s, K, T, r, sigma) for s in S_am])
payoff_plot = np.maximum(K - S_am, 0)

ax.plot(S_am, V_am[:, 0], color=PRIMARY, linewidth=2, label='American put')
ax.plot(S_am, V_eur_put, '--', color=SECONDARY, linewidth=2, label='European put')
ax.plot(S_am, payoff_plot, ':', color='grey', linewidth=1.5, label='Intrinsic value')
ax.fill_between(S_am, V_eur_put, V_am[:, 0], alpha=0.15, color=TERTIARY, label='Early exercise premium')

ax.set_xlim(50, 150)
ax.set_ylim(0, 55)
ax.set_xlabel('Stock Price ($)')
ax.set_ylabel('Put Price ($)')
ax.set_title('American vs European Put')
ax.legend()
plt.tight_layout()
plt.show()

### Interpreting the American vs European Put Plot

Three curves appear in the plot:

1. **Grey dotted line (intrinsic value)**: The payoff $\max(K - S, 0)$. This is the floor -- the
   American put can never be worth less than this.

2. **Dashed orange line (European put)**: The Black-Scholes European put price. Notice that for
   low stock prices, the European put actually dips *below* the intrinsic value. This is because
   the European holder cannot exercise early and must wait, during which time the present value
   of the strike payment decreases.

3. **Solid blue line (American put)**: Always at or above both the European put and the intrinsic
   value. The shaded green region between the American and European curves is the early exercise
   premium.

The premium is largest for deep in-the-money puts (low $S$) where early exercise is most
valuable, and vanishes for out-of-the-money puts (high $S$) where early exercise would never
make sense.

> **Important:** This is a problem that **cannot be solved with the Black-Scholes formula**. There
> is no known closed-form expression for the American put price. The finite difference method with
> PSOR is one of the standard numerical approaches used in practice.> **Key Concept:** The gap between the American and European put prices is the **early exercise premium**. It is largest when the put is deep in the money (low $S$) because that is where the benefit of exercising early — receiving $K$ in cash and investing it at $r$ — is greatest. For deep out-of-the-money puts, the early exercise premium is negligible because you would never exercise an option worth almost nothing.


---

## 11. Greeks from the Grid -- Sensitivities for Free

One of the most powerful advantages of finite difference methods over Monte Carlo is that the
**Greeks come for free**. Once you have computed the option values on a grid, extracting the
sensitivities requires no additional PDE solves -- just simple finite difference formulas applied
to the grid values you already have.

### What Are Greeks?

Greeks measure how the option price changes in response to small changes in the inputs:

| Greek | Symbol | Measures | Finite Difference Formula |
|:------|:-------|:---------|:--------------------------|
| **Delta** | $\Delta$ | Sensitivity to stock price | $\frac{V_{j+1}^0 - V_{j-1}^0}{2\Delta S}$ |
| **Gamma** | $\Gamma$ | Curvature (rate of change of delta) | $\frac{V_{j+1}^0 - 2V_j^0 + V_{j-1}^0}{(\Delta S)^2}$ |
| **Theta** | $\Theta$ | Sensitivity to time (time decay) | $\frac{V_j^1 - V_j^0}{\Delta t}$ |

### How It Works

Consider Delta. The analytical definition is $\Delta = \partial V / \partial S$. On our grid, the
option values at stock prices $S_{j-1}$, $S_j$, and $S_{j+1}$ are already computed. The central
difference approximation gives:

$$\Delta_j \approx \frac{V_{j+1} - V_{j-1}}{2\Delta S}$$

This is second-order accurate -- the same accuracy as the spatial discretization itself. No bumping,
no re-pricing, no additional computation.

For Gamma, we use the second-order central difference:

$$\Gamma_j \approx \frac{V_{j+1} - 2V_j + V_{j-1}}{(\Delta S)^2}$$

For Theta, we use the first time step:

$$\Theta_j \approx \frac{V_j^1 - V_j^0}{\Delta t}$$

where $V_j^0$ is today's value and $V_j^1$ is the value one time step into the future.

> **Key Concept:** With finite differences, Greeks are **free**: you already have option values at
> neighboring grid points, so computing Delta, Gamma, and Theta is just arithmetic on the existing
> grid. Compare this to Monte Carlo, where computing Greeks requires either bumping and re-simulating
> (expensive) or using variance reduction techniques like pathwise derivatives (complex). This is a
> major practical advantage of PDE methods.

The next cell computes Delta, Gamma, and Theta from the Crank-Nicolson grid and plots them across
a range of stock prices. We use a finer grid ($M = 400$) for smoother Greek curves.

In [ ]:
# Compute Greeks from the CN grid for European call
S_grid_cn, V_cn_call = crank_nicolson_fd(S_max, 400, 500, K, T, r, sigma, 'call')
dS_g = S_grid_cn[1] - S_grid_cn[0]
dt_g = T / 500

V0 = V_cn_call[:, 0]
V1 = V_cn_call[:, 1]

delta = (V0[2:] - V0[:-2]) / (2 * dS_g)
gamma = (V0[2:] - 2 * V0[1:-1] + V0[:-2]) / dS_g**2
theta = (V1[1:-1] - V0[1:-1]) / dt_g
S_inner = S_grid_cn[1:-1]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

mask = (S_inner > 50) & (S_inner < 150)
axes[0].plot(S_inner[mask], delta[mask], color=PRIMARY, linewidth=2)
axes[0].set_title('Delta (Δ)')
axes[0].set_xlabel('Stock Price')

axes[1].plot(S_inner[mask], gamma[mask], color=SECONDARY, linewidth=2)
axes[1].set_title('Gamma (Γ)')
axes[1].set_xlabel('Stock Price')

axes[2].plot(S_inner[mask], theta[mask] / 252, color=TERTIARY, linewidth=2)  # per day
axes[2].set_title('Theta (Θ) per day')
axes[2].set_xlabel('Stock Price')

plt.suptitle('Greeks from FD Grid — European Call', fontsize=14)
plt.tight_layout()
plt.show()

### Interpreting the Greek Plots

**Delta (left panel):** Delta ranges from 0 (far out of the money) to 1 (deep in the money) for a
call. It looks like a smoothed step function centered at the strike price. At the money ($S = 100$),
Delta is approximately 0.5 -- meaning the option price changes by about $0.50 for each $1.00
move in the stock.

**Gamma (middle panel):** Gamma peaks at the money and falls off on both sides. High Gamma means
Delta is changing rapidly -- the option's behavior is most nonlinear near the strike. This is where
hedging is most difficult and most important.

**Theta (right panel):** Theta is negative everywhere for a call (the option loses value as time
passes), with the largest time decay at the money. Deep in- or out-of-the-money options have
little time value to lose, so their Theta is small.

> **Key Concept:** The three Greeks are related by the Black-Scholes PDE itself. At any point:
> $$\Theta + \frac{1}{2}\sigma^2 S^2 \Gamma + rS\Delta - rV = 0$$
> This provides a useful sanity check: if your computed Greeks do not approximately satisfy this
> relationship, something is wrong with your implementation.> **Important:** Computing Greeks via finite differences from the grid is both fast and accurate. You don't need to bump-and-reprice (which requires solving the PDE multiple times). Instead, you get Delta, Gamma, and Theta directly from the grid values you already computed — essentially "for free." This is a major practical advantage of PDE-based pricing over Monte Carlo, where Greeks require either pathwise differentiation or bump-and-reprice.


---

## 12. Convergence Analysis

How fast do our numerical methods converge to the true answer as we refine the grid? This is a
crucial question for practical use: it tells us how fine a grid we need for a given accuracy target.

### What to Expect

- **Implicit method:** First-order in time, second-order in space. The error should decrease roughly
  proportionally to the number of grid points (doubling the grid should halve the error).
- **Crank-Nicolson:** Second-order in both time and space. The error should decrease as the *square*
  of the grid refinement (doubling the grid should quarter the error).

We test both methods on a sequence of increasingly fine grids, measuring the absolute error against
the Black-Scholes analytical price.

The next cell runs the convergence study and produces a log-log plot. On a log-log plot, the slope
of the error curve indicates the convergence order: slope $-1$ means first-order, slope $-2$ means
second-order.

In [ ]:
# Convergence study: error vs grid refinement
M_vals = [50, 100, 200, 400]
N_vals = [50, 200, 500, 2000]

errors_cn = []
errors_im = []
grid_sizes = []

for M_test, N_test in zip(M_vals, N_vals):
    S_g, V_cn_test = crank_nicolson_fd(S_max, M_test, N_test, K, T, r, sigma, 'call')
    S_g2, V_im_test = implicit_fd(S_max, M_test, N_test, K, T, r, sigma, 'call')
    idx = np.argmin(np.abs(S_g - S0))
    idx2 = np.argmin(np.abs(S_g2 - S0))
    errors_cn.append(abs(V_cn_test[idx, 0] - bs_call_price))
    errors_im.append(abs(V_im_test[idx2, 0] - bs_call_price))
    grid_sizes.append(M_test * N_test)

fig, ax = plt.subplots()
ax.loglog(grid_sizes, errors_cn, 'o-', color=TERTIARY, linewidth=2, markersize=8, label='Crank-Nicolson')
ax.loglog(grid_sizes, errors_im, 's-', color=SECONDARY, linewidth=2, markersize=8, label='Implicit')
ax.set_xlabel('Grid points (M × N)')
ax.set_ylabel('Absolute Error')
ax.set_title('Convergence: Error vs Grid Size')
ax.legend()
plt.tight_layout()
plt.show()

print(f"{'M×N':<15} {'CN Error':>12} {'Implicit Error':>15}")
print('-' * 44)
for gs, e_cn, e_im in zip(grid_sizes, errors_cn, errors_im):
    print(f"{gs:<15} {e_cn:>12.6f} {e_im:>15.6f}")

### Interpreting the Convergence Results

The log-log plot confirms our theoretical expectations:

- **Crank-Nicolson** (green) converges faster -- its error drops more steeply as the grid is
  refined. This is the benefit of second-order accuracy in time.
- **Implicit** (orange) converges more slowly, consistent with its first-order time accuracy.

The table shows the concrete numbers. For a given grid size, Crank-Nicolson typically achieves
an error that is several times smaller than the implicit method.

In practice, this means that to achieve a target accuracy (say, $0.01 for pricing), Crank-Nicolson
needs a much coarser (and therefore cheaper) grid than the implicit method. This computational
efficiency is why Crank-Nicolson is the standard choice.

> **Key Concept:** Convergence analysis is not just an academic exercise. In production systems,
> you need to know that your pricer converges to the right answer and how fast. Running the method
> on a sequence of grids and checking that the error decreases at the expected rate is a critical
> validation step. If the convergence rate is wrong, there is a bug.### Practical Convergence Guidelines

| Grid size ($M \times N$) | Typical accuracy | Use case |
|:---|:---|:---|
| $50 \times 50$ | ~1% error | Quick sanity check |
| $200 \times 500$ | ~0.1% error | Production pricing (European) |
| $200 \times 1000$ | ~0.01% error | Production pricing (American) |
| $500 \times 2000$ | ~0.001% error | Risk management, model validation |

> **Common Mistake:** Don't refine only the spatial grid ($M$) without also refining the time grid ($N$). The two must be balanced — otherwise the time discretisation error dominates and you waste computational effort on spatial resolution you can't exploit.


---

## 13. Summary and Takeaways

We have built three finite difference methods from scratch and applied them to both European and
American option pricing. Here are the key lessons:

1. **The Black-Scholes PDE is a heat equation.** All the numerical techniques from physics carry
   over to finance.

2. **The explicit method is simple but dangerous.** It requires very small time steps to avoid
   catastrophic instability (the CFL condition). In practice, it is rarely used.

3. **The implicit method is always stable.** It requires solving a tridiagonal system at each step
   (via the Thomas algorithm), but can use much larger time steps. The trade-off: only first-order
   accuracy in time.

4. **Crank-Nicolson is the industry standard.** By averaging explicit and implicit, it achieves
   second-order accuracy in time while remaining unconditionally stable. Same cost as implicit,
   much better accuracy.

5. **American options require special treatment.** The early exercise constraint turns the PDE into
   a free boundary problem, solved here with Projected SOR.

6. **Greeks come for free.** The finite difference grid contains all the information needed to
   compute Delta, Gamma, and Theta without any additional PDE solves.

## 14. References

1. Wilmott, P., Dewynne, J. & Howison, S. *The Mathematics of Financial Derivatives*, Cambridge, 1995.
2. Duffy, D. J. *Finite Difference Methods in Financial Engineering*, Wiley, 2006.
3. Hull, J. C. *Options, Futures, and Other Derivatives*, 11th ed., Pearson, 2022.
4. Tavella, D. & Randall, C. *Pricing Financial Instruments: The FD Method*, Wiley, 2000.
5. Glasserman, P. *Monte Carlo Methods in Financial Engineering*, Springer, 2003.### When to Use Finite Differences vs Other Methods

| Scenario | Best method | Why |
|:---------|:-----------|:----|
| European vanilla | Analytical (BSM) | Exact formula exists |
| American option (1 underlying) | **Finite differences** or trees | FD handles free boundary well |
| Path-dependent (Asian, lookback) | Monte Carlo | Need full path simulation |
| Multi-asset (3+ underlyings) | Monte Carlo | FD grids explode in high dimensions |
| 1-2 underlyings, need Greeks | **Finite differences** | Greeks come free from the grid |


> **Final note:** Mastery of the concepts in this notebook is essential for the CFA Level 1 exam, as well as for practical financial analysis work. Practice the worked examples by hand and verify your understanding by reproducing the code from scratch.
